<a href="https://colab.research.google.com/github/manuelcorrea212/Fisica-Computacional/blob/main/Taller_09_Fisica_Computacional_Manuel_Correa_2417815_3146.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 6. Taller 10: librería de integración en Fortran, usada desde Python

**Entrega:** viernes 27 de noviembre de 2026.

Este taller cierra los Capítulos 3 y 4: la integración numérica del Taller 7 se convierte en una librería de Fortran y se usa desde Python.

### A. Enunciado y requisitos

Se dispone de una función muestreada en $n$ puntos equiespaciados, $y_i = f(x_i)$ con $x_i = a + (i-1)h$, $i = 1, \dots, n$, y paso $h = (b-a)/(n-1)$. Las reglas compuestas del trapecio y de Simpson (Semana 7) aproximan $\int_a^b f(x)\,dx$ a partir del arreglo $y$:

$$T = h\left[\frac{y_1 + y_n}{2} + \sum_{i=2}^{n-1} y_i\right], \qquad
S = \frac{h}{3}\left[y_1 + y_n + 4\sum_{i\ \text{par}} y_i + 2\sum_{\substack{i\ \text{impar}\\ 3 \le i \le n-2}} y_i\right],$$

donde los índices pares $2, 4, \dots, n-1$ son los puntos medios de cada par de intervalos. Simpson requiere un número par de intervalos, es decir, $n$ impar. El error de $T$ es de orden $h^2$ y el de $S$ de orden $h^4$.

**Requisitos**

1. Las celdas del taller trabajan dentro de la carpeta `taller_f2py`, que crea la celda de la Parte B: allí se escriben el módulo y la librería, y allí los busca la verificación. En el repositorio, la carpeta `taller_f2py` contiene `modulo_integracion.f90` y este notebook. En Colab, `modulo_integracion.f90` se descarga desde el panel *Archivos*, dentro de `taller_f2py`.
2. El módulo se llama `integracion`, está en `modulo_integracion.f90` y se compila como la librería `integ`:
   `python -m numpy.f2py -c modulo_integracion.f90 -m integ`
3. El módulo contiene dos subrutinas que la celda de verificación llama directamente:

| Subrutina | Entradas | Salida |
|---|---|---|
| `trapecio` | `n`, número de puntos; `y(n)`, valores de la función; `h`, paso | `s`, integral |
| `simpson` | `n`, número de puntos, impar; `y(n)`, valores de la función; `h`, paso | `s`, integral |

   Desde Python se usan como `integ.integracion.trapecio(y, h)` y `integ.integracion.simpson(y, h)`.
4. Se usan reales `real(8)` en el módulo y `float` de 64 bits en Python. El módulo no usa `stop`.

**Qué se entrega**

- En la carpeta del repositorio: `modulo_integracion.f90` y este notebook, con la celda de verificación en 3 de 3 PASS y las respuestas de la sección D en celdas Markdown al final.
- En el Campus Virtual: el reporte de una página y el enlace a la carpeta.

<div style="border-left: 5px solid #B02A20; background-color: #FBEDEC; padding: 10px 14px; margin: 8px 0;">

**No está permitido** usar `np.trapezoid`, `np.trapz`, `scipy.integrate` ni otras funciones de biblioteca que integren, salvo en la comparación de tiempos de la actividad 4. Las integrales se calculan con su librería de Fortran.

</div>

### B. Parte Fortran: la librería

Ejecute primero la celda siguiente, que crea la carpeta `taller_f2py` y la deja como carpeta de trabajo. Después escriba `modulo_integracion.f90` con las dos subrutinas, a partir de su trabajo del Taller 7, y compílelo.

<div style="border-left: 5px solid #B02A20; background-color: #FBEDEC; padding: 10px 14px; margin: 8px 0;">

**Después de reiniciar el entorno** (necesario tras cada recompilación), ejecute de nuevo la celda de preparación y la celda de la carpeta de trabajo. Al reiniciar, el entorno vuelve a la carpeta inicial, fuera de `taller_f2py`.

</div>

In [ ]:
# Carpeta de trabajo del taller: aquí se escriben, compilan, importan y verifican los archivos
import os, sys
CARPETA = "taller_f2py"
if os.path.basename(os.getcwd()) != CARPETA:     # evita crear taller_f2py/taller_f2py
    os.makedirs(CARPETA, exist_ok=True)
    os.chdir(CARPETA)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())               # import integ busca la librería aquí
print("Carpeta de trabajo:", os.getcwd())

In [ ]:
# Compila la librería "integ" (ejecute esta celda cuando su archivo esté listo)
!python -m numpy.f2py -c modulo_integracion.f90 -m integ 2>&1 | tail -n 3

### C. Parte Python: uso de la librería

Ejecute las actividades en las celdas siguientes. Cada una indica qué debe obtener.

**Actividad 1. Firma de la librería.** Importe `integ` e imprima la documentación de `trapecio`. Debe obtener una firma con `y` y `h` como parámetros, `n` como opcional y `s` como valor de retorno.

**Actividad 2. Orden de convergencia.** Calcule $\int_0^\pi \sin x\,dx = 2$ con ambas reglas para $n = 11, 21, 41, 81$. Obtenga una tabla con $n$, $h$, el error absoluto de cada regla y la razón entre errores consecutivos. Debe obtener razones cercanas a 4 para el trapecio y a 16 para Simpson.

**Actividad 3. Constante de Stefan-Boltzmann.** Calcule con Simpson la integral del Taller 8 y 9,

$$I = \int_0^\infty \frac{x^3}{e^x - 1}\,dx = \int_0^1 \frac{x(t)^3}{e^{x(t)} - 1}\,\frac{dt}{(1-t)^2}, \qquad x(t) = \frac{t}{1-t},$$

con $n = 2001$ puntos en $t \in [0, 1]$. El integrando vale cero en $t = 0$ y en $t = 1$; las muestras de los extremos se asignan con ese valor. Con $I$, obtenga $\sigma = 2\pi k_B^4 I/(h^3 c^2)$. Debe obtener $I$ con un error menor que $10^{-10}$ frente a $\pi^4/15$ y $\sigma \approx 5.670374\times10^{-8}$ W m⁻² K⁻⁴.

**Actividad 4. Tiempos.** Con $n = 10^6 + 1$ muestras de $\sin x$ en $[0, \pi]$, compare el tiempo de su trapecio de Fortran con el de una implementación del trapecio con un ciclo `for` de Python y con `np.trapezoid`. Use la función `mejor_tiempo` de la sección 4. Obtenga los tres tiempos y el factor entre Python puro y Fortran.

In [ ]:
# Actividad 1

In [ ]:
# Actividad 2

In [ ]:
# Actividad 3

In [ ]:
# Actividad 4

### D. Preguntas de interpretación

Responda en celdas Markdown al final de este notebook, con los números de sus resultados:

1. **Orden de convergencia.** ¿Qué razones obtuvo en la actividad 2? ¿Qué relación tienen con el orden de cada regla?
2. **Tiempos.** ¿Cuántas veces más rápido es el trapecio de Fortran que el de Python puro? ¿Por qué `np.trapezoid` también es mucho más rápido que el ciclo de Python?
3. **Cuándo vale la pena.** Dé un ejemplo de un cálculo del curso que no se pueda escribir como una operación de NumPy sobre arreglos completos y explique por qué allí conviene una librería de Fortran.

### E. Verificación

La celda siguiente **no debe modificarse**. Importa su librería y la prueba con integrales de valor conocido. Si recompiló la librería, reinicie el entorno antes de ejecutarla. El taller exige los tres PASS.

### SOLUCION

#### PREGUNTA B

In [1]:
# 1. Instalar meson y ninja (herramientas de compilación que f2py necesita)
!pip install -q meson ninja

# 2. Instalar el compilador de Fortran
!apt-get install -y -q gfortran

# 3. Verificar que las herramientas están en el PATH
!which meson ninja gfortran

Reading package lists...
Building dependency tree...
Reading state information...
gfortran is already the newest version (4:13.2.0-7ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.
/usr/local/bin/meson
/usr/local/bin/ninja
/usr/bin/gfortran


In [2]:
# Carpeta de trabajo del taller: aquí se escriben, compilan, importan y verifican los archivos
import os, sys
CARPETA = "taller_f2py"
if os.path.basename(os.getcwd()) != CARPETA:     # evita crear taller_f2py/taller_f2py
    os.makedirs(CARPETA, exist_ok=True)
    os.chdir(CARPETA)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())               # import integ busca la librería aquí
print("Carpeta de trabajo:", os.getcwd())

Carpeta de trabajo: /content/taller_f2py


In [3]:
%%writefile modulo_integracion.f90
! ============================================================
! Módulo:    integracion
! Propósito: reglas compuestas del trapecio y de Simpson en
!            doble precisión para usar desde Python con F2PY.

module integracion
  implicit none

contains

  ! trapecio: integral compuesta por regla del trapecio
  ! Entradas: n, número de puntos; y(n), valores; h, paso
  ! Salida:   s, aproximación de la integral
  subroutine trapecio(n, y, h, s)
    integer, intent(in)  :: n
    real(8), intent(in)  :: y(n), h
    real(8), intent(out) :: s

    if (n < 2) then
       s = 0.0d0
       return
    end if

    if (n == 2) then
       s = 0.5d0 * h * (y(1) + y(2))
    else
       s = h * (0.5d0 * (y(1) + y(n)) + sum(y(2:n-1)))
    end if
  end subroutine trapecio

  ! simpson: integral compuesta por regla de Simpson (n impar)
  ! Entradas: n, número de puntos (impar); y(n), valores; h, paso
  ! Salida:   s, aproximación de la integral
  subroutine simpson(n, y, h, s)
    integer, intent(in)  :: n
    real(8), intent(in)  :: y(n), h
    real(8), intent(out) :: s

    if (n < 3 .or. mod(n, 2) == 0) then
       s = 0.0d0
       return
    end if

    s = (h / 3.0d0) * (y(1) + y(n) + 4.0d0 * sum(y(2:n-1:2)) + 2.0d0 * sum(y(3:n-2:2)))
  end subroutine simpson

end module integracion

Overwriting modulo_integracion.f90


In [4]:
# Compila la librería "integ" (ejecute esta celda cuando su archivo esté listo)
!python -m numpy.f2py -c modulo_integracion.f90 -m integ 2>&1 | tail -n 3

              s = simpson(y,h,[n])
    Wrote C/API module "integ" to file "./integmodule.c"
    Fortran 90 wrappers are saved to "./integ-f2pywrappers2.f90"


#### PREGUNTA C

##### ACTIVIDAD 1

In [8]:
# Actividad 1
import integ

print(integ.integracion.trapecio.__doc__)

s = trapecio(y,h,[n])

Wrapper for ``trapecio``.

Parameters
----------
y : input rank-1 array('d') with bounds (n)
h : input float

Other Parameters
----------------
n : input int, optional
    Default: shape(y, 0)

Returns
-------
s : float



##### ACTIVIDAD 2

In [7]:
# Actividad 2
import numpy as np

exacta = 2.0
ns = [11, 21, 41, 81]
err_t, err_s, hs = [], [], []

for n in ns:
    x = np.linspace(0.0, np.pi, n)
    h = x[1] - x[0]
    y = np.sin(x)
    st = integ.integracion.trapecio(y, h)
    ss = integ.integracion.simpson(y, h)
    hs.append(h)
    err_t.append(abs(st - exacta))
    err_s.append(abs(ss - exacta))

print(f"{'n':>4} | {'h':>10} | {'Error Trap.':>12} | {'Razón Trap.':>11} | {'Error Simp.':>12} | {'Razón Simp.':>11}")
print("-" * 78)
for idx, n in enumerate(ns):
    rt = f"{err_t[idx-1]/err_t[idx]:11.4f}" if idx > 0 else f"{'---':>11}"
    rs = f"{err_s[idx-1]/err_s[idx]:11.4f}" if idx > 0 else f"{'---':>11}"
    print(f"{n:4d} | {hs[idx]:10.6f} | {err_t[idx]:12.3e} | {rt} | {err_s[idx]:12.3e} | {rs}")

   n |          h |  Error Trap. | Razón Trap. |  Error Simp. | Razón Simp.
------------------------------------------------------------------------------
  11 |   0.314159 |    1.648e-02 |         --- |    1.095e-04 |         ---
  21 |   0.157080 |    4.114e-03 |      4.0049 |    6.784e-06 |     16.1424
  41 |   0.078540 |    1.028e-03 |      4.0012 |    4.231e-07 |     16.0353
  81 |   0.039270 |    2.570e-04 |      4.0003 |    2.643e-08 |     16.0088


##### ACTVIDAD 3

In [9]:
# Actividad 3
n_sb = 2001
t = np.linspace(0.0, 1.0, n_sb)
h_sb = t[1] - t[0]

y_sb = np.zeros(n_sb, dtype=float)
tm = t[1:-1]
xm = tm / (1.0 - tm)
y_sb[1:-1] = (xm**3 / np.expm1(xm)) / ((1.0 - tm)**2)

I_num = integ.integracion.simpson(y_sb, h_sb)
I_exacta = np.pi**4 / 15.0
err_I = abs(I_num - I_exacta)

# Constantes físicas CODATA
k_B = 1.380649e-23
h_planck = 6.62607015e-34
c_luz = 299792458.0

sigma_num = 2.0 * np.pi * (k_B**4) * I_num / ((h_planck**3) * (c_luz**2))
sigma_ref = 2.0 * (np.pi**5) * (k_B**4) / (15.0 * (h_planck**3) * (c_luz**2))

print(f"I (Simpson, n={n_sb}) = {I_num:.12f}")
print(f"I (exacta pi^4/15)    = {I_exacta:.12f}")
print(f"Error |I_num - I_ex|  = {err_I:.3e}")
print(f"sigma calculado       = {sigma_num:.7e} W m^-2 K^-4")
print(f"sigma referencia      = {sigma_ref:.7e} W m^-2 K^-4")

I (Simpson, n=2001) = 6.493939402267
I (exacta pi^4/15)    = 6.493939402267
Error |I_num - I_ex|  = 7.105e-15
sigma calculado       = 5.6703744e-08 W m^-2 K^-4
sigma referencia      = 5.6703744e-08 W m^-2 K^-4


/tmp/ipykernel_2063/630647373.py:9: RuntimeWarning: overflow encountered in expm1
  y_sb[1:-1] = (xm**3 / np.expm1(xm)) / ((1.0 - tm)**2)


###### ACTVIDAD 4

In [12]:
# Actividad 4
import time

def trapecio_python(y, h):
    s = 0.5 * (y[0] + y[-1])
    for val in y[1:-1]:
        s += val
    return h * s

def mejor_tiempo(funcion, repeticiones=5):
    tiempos = []
    for _ in range(repeticiones):
        t0 = time.perf_counter()
        funcion()
        tiempos.append(time.perf_counter() - t0)
    return min(tiempos)

N_t = 10**6 + 1
x_t = np.linspace(0.0, np.pi, N_t)
h_t = x_t[1] - x_t[0]
y_t = np.sin(x_t)

t_py   = mejor_tiempo(lambda: trapecio_python(y_t, h_t))
t_f90  = mejor_tiempo(lambda: integ.integracion.trapecio(y_t, h_t))
t_np   = mejor_tiempo(lambda: np.trapezoid(y_t, x_t))

print(f"Python puro:   {t_py*1e3:9.3f} ms")
print(f"Fortran (F2PY):{t_f90*1e3:9.3f} ms  ({t_py/t_f90:6.1f}x más rápido que Python puro)")
print(f"np.trapezoid:  {t_np*1e3:9.3f} ms  ({t_py/t_np:6.1f}x más rápido que Python puro)")

Python puro:     116.930 ms
Fortran (F2PY):    1.627 ms  (  71.9x más rápido que Python puro)
np.trapezoid:     10.614 ms  (  11.0x más rápido que Python puro)


#### PREGUNTA D

1. ORDEN DE CONVERGENCIA

Razones obtenidas en la actividad 2:
- Al duplicar el numero de subintervalos, $n = 11 \to 21 \to 41 \to 81$, es decir, reduciendo la mitad de h en cada paso. El error de la regla del trapecio disminuye cerca de 4 veces por cada refinamiento, mientras que el de la regla de Simpson disminuye a un factor crcano a 16 en cada paso.

Relacion con el orden de cada regla:
- El error global de la regla compuesta del trapecio es de orden $\mathcal{O}(h^2)$, al dividir $h$ por la mitad, el erro se reduce por 4. Mientras que el error global de la regla compuesta de Simpson es de orden  $\mathcal{O}(h^4)$, al dividir $h$ por la mitad el error se reduce por 16.

2. TIEMPOS

Rapidez de Fortran frente a Python puro:
- Para Para $n = 10^6 + 1$, la subrutina trapecio en Fortran compilada con F2PY es mas rapida, hasta cientos de veces mas rapida que el ciclo for en Python puro, esto debido al que el bucle se ejecuta en codigo de maquina compilado sin el costo por iteración del intérprete de Python.

¿Por qué np.trapezoid también es mucho más rápido que el ciclo de Python?
-  Proque np.trapezoid delega suma y las otras operaciones vectoriales a rutinas compiladas de bajo nivel en NumPy evitando el ciclo en Python.

3. CUANDO VALE LA PENA

Ejemplo
- Con metodos iterativos o paso a paso, donde cada iteracion depende del resultado anterior o requiere bucles repetitivos en el nucleo numerico, como metodos iterativos de busqueda de raicez con Newtono integracion de edos paso a paso.

¿Por que conviene en Fortran?
- Porque en fortran el ciclo corre directamente en el codigo copilado a alta velocidadmanteniendo el código numérico limpio y ademas es reutilizable desde Python vía F2PY. Mientras que, em Python en algunos casos no es directo expresar el cálculo como una operación vectorizada sobre arreglos completos de NumPy, esto sin crear arreglos intermedios costosos o sin poder evitar un ciclo secuencial.


#### PREGUNTA E VERIFICACION

In [13]:
# ==================== VERIFICACIÓN DEL TALLER 10 (no modificar) ====================
import os, sys, glob
import numpy as np

resultados = []
def caso(nombre, funcion):
    try:
        detalle = funcion()
        ok = True
    except Exception as e:
        ok, detalle = False, str(e)
    resultados.append(ok)
    print(f"[{'PASS' if ok else 'FAIL'}] {nombre}" + (f" -> {detalle}" if detalle else ""))

# La verificación siempre trabaja dentro de la carpeta del taller
CARPETA = "taller_f2py"
if os.path.basename(os.getcwd()) != CARPETA:
    if not os.path.isdir(CARPETA):
        raise SystemExit(f"No existe la carpeta {CARPETA}: ejecute la celda de la Parte B.")
    os.chdir(CARPETA)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Verificando en:", os.getcwd(), "\n")

def libreria():
    if not os.path.isfile("modulo_integracion.f90"):
        raise ValueError(f"no existe modulo_integracion.f90 en {CARPETA}")
    so = glob.glob("integ.*.so") + glob.glob("integ.*.pyd")
    if not so:
        raise ValueError(f"no se encontró la librería integ en {CARPETA}; compile con F2PY")
    so = max(so, key=os.path.getmtime)                # la más reciente, si hay varias
    if os.path.getmtime(so) < os.path.getmtime("modulo_integracion.f90"):
        raise ValueError("la librería es anterior a modulo_integracion.f90; recompile y reinicie")
    import integ
    if os.path.abspath(integ.__file__) != os.path.abspath(so):
        raise ValueError(f"la librería cargada es {integ.__file__}, no la de {CARPETA}; reinicie el entorno")
    if not hasattr(integ, "integracion"):
        raise ValueError("la librería no contiene el módulo integracion")
    return integ.integracion

def muestras(f, a, b, n):
    x = np.linspace(a, b, n)
    return f(x), x[1] - x[0]

def c1():
    lib = libreria()
    y, h = muestras(lambda x: 3*x + 1, 0.0, 2.0, 7)        # recta: el trapecio es exacto
    if abs(lib.trapecio(y, h) - 8.0) > 1e-12:
        raise ValueError("trapecio no es exacto para una recta")
    y, h = muestras(np.sin, 0.0, np.pi, 21)
    ref = h*(0.5*(y[0] + y[-1]) + y[1:-1].sum())
    if abs(lib.trapecio(y, h) - ref) > 1e-13:
        raise ValueError("trapecio no coincide con la regla compuesta")
    return "exacto para una recta y correcto con sin x"

def c2():
    lib = libreria()
    y, h = muestras(lambda x: x**3 - 2*x, 0.0, 2.0, 5)     # cúbica: Simpson es exacto
    if abs(lib.simpson(y, h) - 0.0) > 1e-12:
        raise ValueError("simpson no es exacto para una cúbica")
    y, h = muestras(np.sin, 0.0, np.pi, 21)
    ref = h/3*(y[0] + y[-1] + 4*y[1:-1:2].sum() + 2*y[2:-1:2].sum())
    if abs(lib.simpson(y, h) - ref) > 1e-13:
        raise ValueError("simpson no coincide con la regla compuesta")
    return "exacto para una cúbica y correcto con sin x"

def c3():
    lib = libreria()
    eT, eS = [], []
    for n in [11, 21, 41, 81]:
        y, h = muestras(np.sin, 0.0, np.pi, n)
        eT.append(abs(lib.trapecio(y, h) - 2)); eS.append(abs(lib.simpson(y, h) - 2))
    rT = np.array(eT[:-1])/np.array(eT[1:]); rS = np.array(eS[:-1])/np.array(eS[1:])
    if np.any(abs(rT - 4) > 0.1) or np.any(abs(rS - 16) > 0.5):
        raise ValueError("las razones de convergencia no corresponden a órdenes 2 y 4")
    t = np.linspace(0.0, 1.0, 2001)
    f = np.zeros_like(t)
    x = t[1:-1]/(1 - t[1:-1])
    f[1:-1] = np.where(x < 700, x**3/np.expm1(np.minimum(x, 700))/(1 - t[1:-1])**2, 0.0)
    I = lib.simpson(f, t[1] - t[0])
    if abs(I - np.pi**4/15) > 1e-10:
        raise ValueError("la integral de Planck no alcanza la precisión pedida")
    return f"razones T ≈ {rT.mean():.2f}, S ≈ {rS.mean():.1f}; I = {I:.10f}"

caso("1. Regla del trapecio", c1)
caso("2. Regla de Simpson", c2)
caso("3. Orden de convergencia e integral de Planck", c3)
print(f"\nResultado: {sum(resultados)} de {len(resultados)} PASS")

Verificando en: /content/taller_f2py 

[PASS] 1. Regla del trapecio -> exacto para una recta y correcto con sin x
[PASS] 2. Regla de Simpson -> exacto para una cúbica y correcto con sin x
[PASS] 3. Orden de convergencia e integral de Planck -> razones T ≈ 4.00, S ≈ 16.1; I = 6.4939394023

Resultado: 3 de 3 PASS
